In [ ]:
# from typing import TypedDict, Literal 

# # Define the structure for email classification
# class EmailClassification(TypedDict):
#     intent: Literal["question", "bug", "billing", "feature", "complex"]
#     urgency: Literal["low", "medium", "high", "critical"]
#     topic: str 
#     summary: str 

# class EmailAgentState(TypedDict):
#     # Raw email data 
#     email_content: str 
#     sender_email: str 
#     email_id: str 

#     # Classification result
#     classification: EmailClassification | None 

#     # Raw search/API results
#     search_results: list[str] | None 
#     customer_history: dict | None

#     # Generated content 
#     draft_response: str | None 
#     messages: list[str] | None


# from langgraph.types import RetryPolicy


# # Agent nodes 
# # Read and classify nodes
# from typing import Literal 
# from langgraph.graph import StateGraph, START, END 
# from langgraph.types import interrupt, Command, RetryPolicy
# from langchain_openai import ChatOpenAI
# from langchain.messages import HumanMessage

# llm = ChatOpenAI(model="gpt-5-nano")


# def read_email(state: EmailAgentState) -> dict:
#     """Extract and parse email content"""
#     # In production, this would connect to your email service
#     return {
#         "messages": [HumanMessage(content=f"Processing email: {state['email_content']}")]
#     }


# def classify_intent(state: EmailAgentState) -> Command[Literal[
#     "search_documentation", "human_review", "draft_response", "bug_tracking"]]:
#     """Use LLM to classify email intent and urgency, then route accordingly"""

#     # Create structured LLM that returns EmailClssification dict
#     structured_llm = llm.with_structured_output(EmailClassification)

#     # Format the prompt on-demand, not stored in state
#     classification_prompt = f"""
#     Analyze this customer email and classify it:

#     Email: {state['email_content']}
#     From:  {state['sender_email']}

#     Provide classification including intent, urgency, topic, and summary. 
#     """

#     # Get structured response directly as dict 
#     classification = structured_llm.invoke(classification_prompt)

#     # Determine next node based on classification
#     if classification['intent'] == 'billing' or classification['urgency'] == 'critical':
#         goto = "human_review"
#     elif classification['intent'] in ['question', 'feature']:
#         goto = "search_documentation"
#     elif classification['intent'] == 'bug':
#         goto = "bug_tracking"
#     else:
#         goto = "draft_response"
    
#     # Store classification as a single dict in state
#     return Command(
#         update={"classification": classification},
#         goto = goto
#     )

# def search_documentation(state: EmailAgentState) -> Command[Literal["draft_response"]]:
#     """Search knowledge base for relevant information"""

#     # Build search query from classification
#     classification = state.get('classification', {})
#     query = f"{classification.get('intent', '')} {classification.get('topic', '')}"

#     try:
#         # Implement your search logic here
#         # Store raw search results, not formatted text 
#         search_results = [
#             "Reset password via Settings > Security > Change Password",
#             "Password must be at least 12 characters",
#             "Include uppercase, lowercase, numbers, and symbols"
#         ]
#     except Exception as e:
#         search_results = [f"Search temproarily unavailable: {str(e)}"]

#     return Command(
#         update={"search_results": search_results}, 
#         goto="draft_response"
#     )

# def bug_tracking(state: EmailAgentState) -> Command[Literal["draft_response"]]:
#     """Create or update bug tracking ticket"""
#     # Create ticket in your bug tracking system

#     ticket_id = "BUG-12345"

#     return Command( 
#         update={
#             "search_results": [f"Bug ticket {ticket_id} created"],
#             "current_step": "bug_tracked"
#         },
#         goto="draft_response"
#     )

# def draft_response(state: EmailAgentState) -> Command[Literal['human_review', 'send_reply']]:
#     """Generate response using context and route based on quality"""
#     classification = state.get('classification', {})

#     # Format context from raw state data on-demand
#     context_sections = []

#     if state.get('search_results'):
#         # Format search results for the prompt
#         formatted_docs = "\n".join([f"- {doc}" for doc in state['search_results']])
#         context_sections.append(f"Relevant documentation:\n{formatted_docs}")

#     if state.get('customer_history'):
#         # Format customer data for the prompt
#         context_sections.append(f"Customer tier: {state['customer_history'].get('tier', 'standard')}")

#     # Build the prompt with formatted context
#     draft_prompt = f"""
#         Draft a response to this customer email:
#         {state['email_content']}

#         Email intent: {classification.get('intent', 'unknown')}
#         Urgency level: {classification.get('urgency', 'medium')}

#         {chr(10).join(context_sections)}

#         Guidelines:
#         - Be professional and helpful
#         - Address their specific concern
#         - Use the provided documentation when relevant
#     """

#     response = llm.invoke(draft_prompt)

#     # Determine if human review needed based on urgency and intent
#     needs_review = (
#         classification.get('urgency') in ['high', 'critical'] or
#         classification.get('intent') == 'complex'
#     )

#     # Route to appropriate next node 
#     goto = "human_review" if needs_review else "send_reply"

#     return Command(
#         update={"draft_response": response.content},
#         goto=goto
#     )

# def human_review(state: EmailAgentState) -> Command[Literal["send_reply", END]]:
#     """Pause for human review using interrupt and route based on decision"""

#     classification = state.get('classification', {})
#     # interrupt() must come first - any code before it will re-run on resume
#     human_decision = interrupt({
#         "email_id": state.get('email_id',''),
#         "original_email": state.get('email_content',''),
#         "draft_response": state.get('draft_response',''),
#         "urgency": classification.get('urgency'),
#         "intent": classification.get('intent'),
#         "action": "Please review and approve/edit this response"
#     })

#     if human_decision.get("approved"):
#         return Command(
#             update={"draft_response": human_decision.get("edited_response", state.get('draft_response',''))},
#             goto="send_reply"
#         )
#     else:
#         return Command(update={}, goto=END)
    
# def send_reply(state: EmailAgentState) -> dict:
#     """Send the email response"""
#     # Integrate with email service
#     print(f"Sending reply: {state['draft_response'][:100]}...")
#     return {}

# from langgraph.checkpoint.memory import MemorySaver
# from langgraph.types import RetryPolicy

# # Create the graph
# workflow = StateGraph(EmailAgentState)

# # Add nodes with appropriate error handling
# workflow.add_node("read_email", read_email)
# workflow.add_node("classify_intent", classify_intent)

# # Add retry policy for nodes that might have transient failures
# workflow.add_node(
#     "search_documentation",
#     search_documentation, 
#     retry_policy=RetryPolicy(max_attempts=3)
# )
# workflow.add_node("bug_tracking", bug_tracking)
# workflow.add_node("draft_response", draft_response)
# workflow.add_node("human_review", human_review)
# workflow.add_node("send_reply", send_reply)

# # Add only the essential edges
# workflow.add_edge(START, "read_email")
# workflow.add_edge("read_email", "classify_intent")
# workflow.add_edge("send_reply", END)

# # Compile with checkpointer for persistence, in case run graph with Local_Server --> Please compile without checkpointer
# memory = MemorySaver()
# app = workflow.compile(checkpointer=memory)

# from typing_extensions import TypedDict 
# from langgraph.graph import StateGraph, START, END 
# from IPython.display import Image, display 

# # ~ Prompt Chaining

# class State(TypedDict):
#     topic: str 
#     joke: str 
#     improved_joke: str 
#     final_joke: str


# def generate_joke(state: State):
#     """First LLM call to generate initial joke"""
#     msg = llm.invoke(f"Write a short joke about {state['topic']}") 
#     return {"joke": msg.content}

# def check_punchline(state: State):
#     """Gate function to check if the joke has a punchline"""

#     # Simple check - does the joke contain "?" or "!"
#     if "?" in state["joke"] or "!" in state["joke"]:
#         return "Pass"
#     return "Fail"


# def improve_joke(state: State):
#     """Second LLM call to improve the joke"""

#     msg = llm.invoke(f"Make this joke funnier by adding wordplay: {state["joke"]}")
#     return {"improved_joke": msg.content}

# def polish_joke(state: State):
#     """Third LLM call for final polish"""
#     msg = llm.invoke(f"Add a surprising twist to this joke: {state['improved_joke']}")
#     return {"final_joke": msg.content}


# # Build workflow
# workflow = StateGraph(State)

# # Add nodes
# workflow.add_node("generate_joke", generate_joke)
# workflow.add_node("improve_joke", improve_joke)
# workflow.add_node("polish_joke", polish_joke)

# # Add edges to connect nodes
# workflow.add_edge(START, "generate_joke")
# workflow.add_conditional_edges(
#     "generate_joke", check_punchline, {"Fail": "improve_joke", "Pass": END}
# )
# workflow.add_edge("improve_joke", "polish_joke")
# workflow.add_edge("plish_joke", END)

# # Compile 
# chain = workflow.compile()

# # Show workflow
# display(Image(chain.get_graph().draw_mermaid_png()))

# # Parallelization
# """
# With parallelization, LLMs work simultaneously on a task. This is either done by running
# multiple independent subtasks at the same time, or running the same task multiple times to check for different outputs. Paralleliztion is commonly used to: 
# # ~ Split up subtasks and run them in parallel, which increase speed
# # ~ Run tasks multiple times to check for diffenet outputs,  
# """
# # Graph state
# class State(TypedDict):
#     topic: str 
#     joke: str 
#     story: str 
#     poem: str 
#     combined_output: str

# # Nodes
# def call_llm_1(state: State):
#     """First LLM call to generate initial joke"""
#     msg = llm.invoke(f"Write a story about {state['topic']}")
#     return {"story": msg.content}

# def call_llm_2(state: State):
#     """Second LLM call to generate story"""
#     msg = llm.invoke(f"Write a story about {state['topic']}")
#     return {"story": msg.content}

# def call_llm_3(state: State):
#     """Third LLM call to generate poem"""
#     msg = llm.invoke(f"Write a poem about {state['topic']}")
#     return {"poem": msg.content}

# def aggregator(state: State):
#     """Combine the joke, story and poem into a single output"""

#     combined = f"Here's a story, joke, and poem about {state['topic']}!\n\n"
#     combined += f"STORY:\n{state['story']}\n\n"
#     combined += f"JOKE:\n{state['joke']}\n\n"
#     combined += f"POEM:\n{state['poem']}"
#     return {"combined_output": combined}

# # Build workflow
# parallel_builder = StateGraph(State)

# # Add nodes
# parallel_builder.add_node("call_llm_1", call_llm_1)
# parallel_builder.add_node("call_llm_2", call_llm_2)
# parallel_builder.add_node("call_llm_3", call_llm_3)
# parallel_builder.add_node("aggregator", aggregator)

# # Add edges to connect nodes
# parallel_builder.add_edge(START, "call_llm_1")
# parallel_builder.add_edge(START, "call_llm_2")
# parallel_builder.add_edge(START, "call_llm_3")
# parallel_builder.add_edge("call_llm_1", "aggregator")
# parallel_builder.add_edge("call_llm_2", "aggregator")
# parallel_builder.add_edge("call_llm_3", "aggregator")
# parallel_builder.add_edge("aggregator", END)
# parallel_workflow = parallel_builder.compile()

# # Show workflow
# display(Image(parallel_workflow.get_graph().draw_mermaid_png()))

# # Invoke
# state = parallel_workflow.invoke({"topic": "cats"})
# print(state["combined_output"])

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END 
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
import operator

# 1️⃣ Define shared state
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

# 2️⃣ Initialize LLM
llm = ChatOpenAI(model="gpt-5.2", temperature=0)

# 3️⃣ Define Agents
def planner_agent(state: AgentState):
    print("🧠 Planner running...")
    response = llm.invoke([
        HumanMessage(content=f"""
        Break this question into research steps:
        {state['messages'][-1].content}
        """)
    ])
    return {"messages": [response]}

def researcher_agent(state: AgentState):
    print("🔎 Researcher running...")
    response = llm.invoke([
        HumanMessage(content=f"""
        Perform research based on this plan:
        {state['messages'][-1].content}
        Provide detailed findings.
        """)
    ])
    return {"messages": [response]}

def writer_agent(state: AgentState):
    print("✍️ Writer running...")
    response = llm.invoke([
        HumanMessage(content=f"""
        Write a clear final answer using:
        {state['messages'][-1].content}
        """)
    ])
    return {"messages": [response]}

def router(state: AgentState):
    last_message = state["messages"][-1].content 
    if "need more data" in last_message:
        return "researcher"
    else:
        return "writer"
    


# 4️⃣ Build Graph
builder = StateGraph(AgentState)

builder.add_node("planner", planner_agent)
builder.add_node("researcher", researcher_agent)
builder.add_node("writer", writer_agent)

builder.set_entry_point("planner")

builder.add_edge("planner", "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", END)
builder.add_conditional_edges(
    "researcher",
    router, 
    {
        "researcher": "researcher",
        "writer": "writer"
    }
)

graph = builder.compile()

# 5️⃣ Run

result = graph.invoke({
    "messages": [HumanMessage(content="How does RAG improve ASR WER ?")]
})

print(result["messages"][-1].content)


# ~ Step 1: Define Shared State
from typing import TypedDict, List, Annotated 
import operator 

class RAGState(TypedDict):
    question: str 
    search_queries: List[str]
    documents: List[str]
    draft_answer: str 
    final_answer: str 

# 1️⃣ Planner Agent
def planner_agent(state: RAGState):
    response = llm.invoke(f"""
    Expand the following research question into 3 search queries:
    {state['question']}
    """)
    queries = response.content.split("\n")
    return {"search_queries": queries}

# 2️⃣ Retriever Agent
def retriever_agent(state: RAGState):
    docs = []

    for query in state["search_quries"]:
        results = vectorstore.similarity_search(query, k = 3)
        docs.extend([r.page_content for r in results])
    
    return {"documents": docs}

# 3️⃣ Evaluator Agent (Self-Reflection)

🧠 Planner running...
🔎 Researcher running...
✍️ Writer running...
## 1) Goal & definitions (what “RAG helps WER” can mean)

### RAG (in ASR context)
In ASR, “RAG” is used in at least three operationally different ways:

1. **Retrieval-augmented decoding / LM integration (during decoding)**  
   Retrieval provides *textual context* (names, terms, prior dialog, catalog entries, etc.) that influences the language model probability during beam search (or rescoring). This includes **kNN-LM**, **cache LMs**, and **contextual biasing with learned fusion**.  
   *This is the most direct way RAG can change WER while remaining acoustically grounded.*

2. **Retrieval-augmented rescoring (after first-pass decoding)**  
   Generate N-best hypotheses from ASR, then rescore using a retrieval-conditioned LM (or LLM) plus optional constraints.  
   *Often yields gains with less risk than full post-editing because it chooses among acoustically plausible candidates.*

3. **Retrieval-augmented post-editin

In [ ]:
from langgraph.graph import StateGraph 
from typing import TypedDict, List, Dict, Any 
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import END
from pinecone import Pinecone 
import os 


llm = ChatOpenAI(model="gpt-5.2")

# ----------- Initialize ----------------------
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("medical-documents")
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# ----------- Define Shared State --------------
class AgentState(TypedDict):
    question: str 
    plan: str 
    documents: List[Dict[str, Any]]
    draft: str 
    final_answer: str
    critique_score: float 


# ------------- Agents --------------------------
def planner(state: AgentState):
    prompt = f"Create a research plan for: {state['question']}"
    plan = llm.invoke(prompt).content
    return {"plan": plan}

def retriever(state: AgentState):
    # Simulate Pinecone retrieval
    retrieved_docs = f"Retrieved documents about: {state['question']}"
    return {"documents": retrieved_docs} 

def pinecone_retriever(state: AgentState):
    query = state['question']

    # 1. Embed query
    query_vector = embedding_model.embed_query(query)

    # 2. Search Pinecone
    results = index.query( 
        vector=query_vector,
        top_k=5,
        include_metadata=True
    )

    # 3. Format results
    documents = []
    for match in results["matches"]:
        documents.append({
            "score": match["score"],
            "text": match["metadata"]["text"], 
            "source": match["metadata"].get("source", "unkown")
        })
    return { 
        "documents": documents
    }

def researcher(state: AgentState):
    # Format documents cleanly
    formatted_docs = "\n\n".join([
        f"[Source: {doc['source']} | Score: {doc['score']:.3f}]\n{doc['text']}"
        for doc in state['documents']
    ])

    prompt = f""" 
    You are a research synthesis expert.

    Use the research plan:
    {state['plan']}

    Use the following retrieved evidence:
    {formatted_docs}

    Instructions:
    - Only use information from retrieved documents.
    - Cite sources in brackets.
    - Do not hallucinate.
    - Provide structured answer.

    Write a detailed response.
    """ 
    
    draft = llm.invoke(prompt).content 
    return {"draft": draft}

    
def critic(state: AgentState):
    prompt = f"""
    You are a strict research evaluator.
    Evaluate the following answer:

    {state['draft']}

    Score it from 0 to 1 based on:
    - Factual grounding in sources
    - Logical consistency
    - Completness

    Return JSON:
    {{
        "score": float,
        "improved_answer": "string"
    }}
    """
    response = llm.invoke(prompt).content 

    # naive parsing (production should use structured output parser)
    import json 
    result = json.loads(response)
    return {
        'critique_score': result['score'],
        'final_answer': result['improved_answer']
    }

def route_after_critic(state: AgentState):
    if state["critique_score"] < 0.8:
        return "researcher"
    return END 




# ------- Build Graph -------------
builder = StateGraph(AgentState)

builder.add_node("planner", planner)
builder.add_node("pinecone_retriever", pinecone_retriever)
builder.add_node("researcher", researcher)
builder.add_node("critic", critic)

builder.set_entry_point("planner")
builder.add_edge("planner", "pinecone_retriever")
builder.add_edge("pinecone_retriever", "researcher")
builder.add_edge("researcher", "critic")
builder.add_conditional_edges("critic", route_after_critic)

builder.set_finish_point("critic")
graph = builder.compile()

# ----------- Run ------------
result = graph.invoke({
    "question": "What symptoms usually appears in Hematology ?"
})

print(result)


# ~ Important Design Insight
# ! Multi-agent 
# & State (Shared Memory)
# & Role separation
# & Controlled routing
# & Self-evaluation
